# 6. Balanced assignment with named workers and jobs

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 20–23. The same model as demo 5,
but the index sets have names (`workers`, `jobs`) and the table is called `time`, so the code reads like the
maths on page 22.

**The problem.** Four workers must do four jobs, one job each. The table gives the time each worker needs for
each job (page 20):

| | Job 1 | Job 2 | Job 3 | Job 4 |
| --- | --- | --- | --- | --- |
| Worker 1 | 9 | 2 | 7 | 8 |
| Worker 2 | 6 | 4 | 3 | 7 |
| Worker 3 | 5 | 8 | 1 | 8 |
| Worker 4 | 7 | 6 | 9 | 4 |

Minimise the total time. $x_{uv} = 1$ if worker $u$ does job $v$ (page 22):

minimise $\sum_{u \in W} \sum_{v \in J} c_{uv} x_{uv}$ subject to $\sum_{v \in J} x_{uv} = 1$ for every worker $u$,
$\sum_{u \in W} x_{uv} = 1$ for every job $v$, $x_{uv} \in \{0, 1\}$.

"Pick one cell in every row and every column": a perfect matching in the bipartite graph of workers and jobs.
There are $4! = 24$ possible assignments.

Workers and jobs are still numbered from 0 in the code: `x[worker, job]` with `worker = 0` for worker 1.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Total time **13**: worker 1 → job 2, worker 2 → job 1, worker 3 → job 3, worker 4 → job 4
(`x[0,1]`, `x[1,0]`, `x[2,2]`, `x[3,3]` = 1), the same as demo 5.

*Original script: `5 Source Files/Python/balanced_assignment_named.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model, name the two index sets and add one binary variable per (worker, job) pair. `time[worker][job]` is the table above.

In [ ]:
# Create the model and the binary variables
model = ConcreteModel()
# x[worker, job]: first index = worker (row of the table), second index = job (column)
workers = range(4)
jobs = range(4)
model.x = Var(workers, jobs, within=Binary)

time = [   # time[worker][job], rows = workers 1-4, columns = jobs 1-4
    [9, 2, 7, 8],
    [6, 4, 3, 7],
    [5, 8, 1, 8],
    [7, 6, 9, 4]
]

The objective: the total time of the chosen pairs, minimised.

In [ ]:
# Create the optimization problem
# minimise the total time of the chosen (worker, job) pairs
model.obj = Objective(expr=sum(sum(time[worker][job] * model.x[worker, job] for job in jobs) for worker in workers), sense=minimize)

Two empty constraint lists, filled in the next cell.

In [ ]:
# Define the equality constraints for workers and jobs
model.worker_constraints = ConstraintList()
model.job_constraints = ConstraintList()

The loops add one constraint per worker (exactly one job each) and one per job (exactly one worker each).

In [ ]:
# Add the constraints in loops: one per worker, then one per job
for worker in workers:      # for each worker
    model.worker_constraints.add(sum(model.x[worker, job] for job in jobs) == 1)     # only assign a single job to a worker
    
for job in jobs:	    # for each job
    model.job_constraints.add(sum(model.x[worker, job] for worker in workers) == 1)  # only assign a single worker to a job

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: the same 8 constraints as demo 5.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the assignment (variables with value 1) and the total time.

In [ ]:
# Display the results
model.display()

## What do you notice?

Put this notebook next to demo 5. What is different, and what is the same?

<details><summary>Answer</summary>

Only the names: `range(4)` became `workers` and `jobs`, `w` became `time`, and `i`, `j` became `worker`, `job`.
The model, the constraints and the answer (13) are identical. Names cost nothing to the solver but make the code
easier to check against the formulation, and the next version (demo 7) takes the same idea further.
</details>